In [ ]:
#!pip install qnexus

In [1]:
import qnexus as qnx
from pytket import Circuit
from datetime import datetime

# 0. ローカル環境からqnexusを利用したNexusアクセス設定

## ID、パスワードを入力してNexusにアクセス

In [2]:
#qnx.login()
#qnx.login_with_credentials()

# 1. Nexusにあるprojects, teamsを確認

In [3]:
all_my_projects = qnx.projects.get_all()

In [4]:
all_my_projects.df()

""


In [5]:
all_my_projects.count()

0

In [6]:
all_my_teams = qnx.teams.get_all()

In [7]:
all_my_teams.df()

""


# 2. NexusのprojectsでJobを管理する

## a. TKETで量子回路を作成する

In [8]:
from pytket import Circuit

In [9]:
circ = Circuit(2)
circ.H(0).CX(0,1)
circ.measure_all()

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

## b. Nexusのprojectsを指定/作成し、アクティベーションする

In [35]:
my_project_ref = qnx.projects.get_or_create("My Project")# input your project
qnx.context.set_active_project(my_project_ref)

## c. projectsにTKET量子回路を保存し、量子回路を最適化し、実機/エミュレータで実行する

### 作成した量子回路をprojectsに保存

In [36]:
my_circ = qnx.circuits.upload(name="my_circ", circuit=circ)

In [37]:
my_circ.download_circuit()

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [38]:
from pytket.circuit.display import render_circuit_jupyter
render_circuit_jupyter(my_circ.download_circuit())

### 作成した量子回路を量子デバイスで実行するために最適化

In [39]:
device_name = "H2-Emulator"
my_quantinuum_config = qnx.QuantinuumConfig(
    device_name=device_name,
)

In [40]:
my_comp = qnx.start_compile_job(
    name = f"my_circ compilation {datetime.now()}",
    programs=[my_circ],#複数の量子回路をコンパイルすることが可能 ex. programs=[my_circ1,my_circ2]
    backend_config=my_quantinuum_config,
    optimisation_level = 3,
)

In [41]:
my_comp.df()

,name,description,created,modified,job_type,last_status,project,backend_config,system,cost,id
0,my_circ compilation 2026-11-10 23:50:47.055180,,2026-11-10 23:50:47.272116+00:00,2026-11-10 23:50:47.272116+00:00,JobType.COMPILE,JobStatusEnum.SUBMITTED,My Project,QuantinuumConfig,Unknown,None,5eef6b8f-893e-4f6e-8303-93dd69d1c1f2


In [42]:
qnx.jobs.status(my_comp)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 50, 50, 594290, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 50, 48, 885597, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 50, 47, 278482, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 50, 49, 156354, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [43]:
qnx.jobs.wait_for(my_comp)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 50, 50, 594290, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 50, 48, 885597, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 50, 47, 278482, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 50, 49, 156354, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [44]:
comp_result = qnx.jobs.results(my_comp)[0].get_output()

In [45]:
comp_circ = comp_result.download_circuit()

In [46]:
comp_circ

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [47]:
render_circuit_jupyter(comp_circ)

### 最適化した量子回路を実機/エミュレータに実行

In [48]:
my_exe = qnx.start_execute_job(
    name = f"my_circ execute {datetime.now()}",
    programs=[comp_result],
    n_shots=[10],#複数の量子回路をコンパイルしている場合には、各回路のショットリストを渡す
#    backend_config=qnx.QuantinuumConfig(device_name="H2-1LE")
    backend_config=qnx.QuantinuumConfig(device_name="H2-Emulator")
)

In [49]:
qnx.jobs.status(my_exe)

JobStatus(status=<JobStatusEnum.RUNNING: 'RUNNING'>, message='The job is running.', error_detail=None, completed_time=None, queued_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 246256, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 211675, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 671471, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [50]:
qnx.jobs.wait_for(my_exe)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 246256, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 211675, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 671471, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)

In [51]:
result = qnx.jobs.results(my_exe)

In [52]:
result[0].download_result().get_counts()

Counter({(1, 1): 6, (0, 0): 4})

In [53]:
qnx.jobs.get_all().df()

,name,description,created,modified,job_type,last_status,project,backend_config,system,cost,id
0,my_circ compilation 2026-11-10 23:50:47.055180,,2026-11-10 23:50:47.272116+00:00,2026-11-10 23:50:50.594290+00:00,JobType.COMPILE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-Emulator,None,5eef6b8f-893e-4f6e-8303-93dd69d1c1f2
1,my_circ execute 2026-11-10 23:51:01.210074,,2026-11-10 23:51:01.436248+00:00,2026-11-10 23:51:04.576273+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-Emulator,None,be4b0f92-cd66-4f89-9442-62e05cdc6432


## d. Nexusデータベースの過去のジョブを抽出する

In [54]:
my_project_ref = qnx.projects.get_or_create("My Project")
qnx.context.set_active_project(my_project_ref)

In [55]:
qnx.jobs.get_all().df()

,name,description,created,modified,job_type,last_status,project,backend_config,system,cost,id
0,my_circ compilation 2026-11-10 23:50:47.055180,,2026-11-10 23:50:47.272116+00:00,2026-11-10 23:50:50.594290+00:00,JobType.COMPILE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-Emulator,None,5eef6b8f-893e-4f6e-8303-93dd69d1c1f2
1,my_circ execute 2026-11-10 23:51:01.210074,,2026-11-10 23:51:01.436248+00:00,2026-11-10 23:51:04.576273+00:00,JobType.EXECUTE,JobStatusEnum.COMPLETED,My Project,QuantinuumConfig,H2-Emulator,None,be4b0f92-cd66-4f89-9442-62e05cdc6432


#### NexusデータベースからJob（コンパイルジョブ）を抽出

In [56]:
jobref = qnx.jobs.get(id="5eef6b8f-893e-4f6e-8303-93dd69d1c1f2")
jobref

CompileJobRef(id=UUID('5eef6b8f-893e-4f6e-8303-93dd69d1c1f2'), annotations=Annotations(name='my_circ compilation 2026-11-10 23:50:47.055180', description='', properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 50, 47, 272116, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 50, 50, 594290, tzinfo=TzInfo(0))), job_type=<JobType.COMPILE: 'compile'>, last_status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, last_message='The job is completed.', last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 50, 50, 594290, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 50, 48, 885597, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 50, 47, 278482, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 50, 49, 156354, tzinfo=datetime.timezone.utc), cancelled_ti

In [57]:
jobcomp = qnx.jobs.results(jobref)[0]
jobcomp

CompilationResultRef(id=UUID('b8950c13-3fb2-407e-9cf9-ae758e17874b'), annotations=Annotations(name='my_circ-compilation', description='', properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 50, 50, 495383, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 50, 57, 899376, tzinfo=TzInfo(0))), project=ProjectRef(id=UUID('31c4325d-6dd9-469a-82ab-8af8b7a44a72'), annotations=Annotations(name='My Project', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 49, 19, 659342, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 49, 19, 659342, tzinfo=TzInfo(0))), contents_modified=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=TzInfo(0)), archived=False, type='ProjectRef'), last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='Program has completed', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 50, 50, 359751, tzinfo=datetime.timezone.utc), queued_time

In [58]:
jobcomp.get_input().download_circuit()

[H q[0]; CX q[0], q[1]; Measure q[0] --> c[0]; Measure q[1] --> c[1]; ]

In [59]:
jobcomp.get_output().download_circuit()

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

#### NexusデータベースからJob(実行ジョブ)を抽出

In [60]:
jobref = qnx.jobs.get(id="be4b0f92-cd66-4f89-9442-62e05cdc6432")
jobref

ExecuteJobRef(id=UUID('be4b0f92-cd66-4f89-9442-62e05cdc6432'), annotations=Annotations(name='my_circ execute 2026-11-10 23:51:01.210074', description='', properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 51, 1, 436248, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=TzInfo(0))), job_type=<JobType.EXECUTE: 'execute'>, last_status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, last_message='The job is completed.', last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 246256, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 211675, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 11, 10, 23, 51, 3, 671471, tzinfo=datetime.timezone.utc), cancelled_time=None, e

In [61]:
jobres = qnx.jobs.results(jobref)[0]
jobres

ExecutionResultRef(id=UUID('20a42823-8a48-4ff5-9593-4a9aad0fd59e'), annotations=Annotations(name='my_circ execute 2026-11-10 23:51:01.210074', description='', properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 51, 1, 436248, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=TzInfo(0))), project=ProjectRef(id=UUID('31c4325d-6dd9-469a-82ab-8af8b7a44a72'), annotations=Annotations(name='My Project', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 11, 10, 23, 49, 19, 659342, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 11, 10, 23, 49, 19, 659342, tzinfo=TzInfo(0))), contents_modified=datetime.datetime(2026, 11, 10, 23, 51, 4, 576273, tzinfo=TzInfo(0)), archived=False, type='ProjectRef'), result_type=<ResultType.PYTKET: 'PYTKET'>, cost=None, last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='Program has completed', error_detail=None, completed_time=datetime.datetime(202

In [62]:
jobres.get_input().download_circuit()

[PhasedX(3.5, 0.5) q[0]; PhasedX(2.5, 0.5) q[1]; ZZPhase(0.5) q[0], q[1]; Measure q[0] --> c[0]; PhasedX(0.5, 0) q[1]; Measure q[1] --> c[1]; ]

In [63]:
jobres.download_result().get_counts()

Counter({(1, 1): 6, (0, 0): 4})

# Syntax checkerの利用 H2利用の前には必ず実行ください。  
## (Jobが問題なく実行可能か、HQCコストなどの確認)

In [64]:
device_name = "H2-1SC"

config = qnx.QuantinuumConfig(device_name=device_name)

job_name = f"execution-job-qir-{datetime.now()}"
ref_execute_job = qnx.start_execute_job(
    programs=[comp_result],
    n_shots=[1000],
    backend_config=config,
    name=job_name,
)

qnx.jobs.wait_for(ref_execute_job)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 11, 10, 23, 52, 8, 915929, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 11, 10, 23, 52, 4, 555689, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 11, 10, 23, 52, 0, 551306, tzinfo=datetime.timezone.utc), running_time=None, cancelled_time=None, error_time=None, queue_position=None, cost=11.6)

## HQCの算出だけであれば、関数`cost`で可能

In [65]:
qnx.client.circuits.cost(comp_result,n_shots=1000,backend_config=config)

11.6